
# Bangla News Cross-Source Comparison
## Simple Zero-Shot Evaluation with Gemini, Qwen and Gemma

### Goal
For the **same incident**, compare how **different Bangla news portals** portray the news.

Pipeline:

**Dataset → Incident Grouping → Cross-Source Pairs → Gemini Reference → Qwen Zero-Shot → Gemma Zero-Shot → BLEU + BERTScore + ROUGE → Average Scores**

### Models
- **Reference:** `gemini-3.6-flash`
- **Qwen:** `Qwen/Qwen3-1.7B`
- **Gemma:** `google/gemma-3-4b-it`

### Final result
For every selected news pair, calculate:
- BLEU
- BERTScore Precision / Recall / F1
- ROUGE-1 F1
- ROUGE-2 F1
- ROUGE-L F1

Then calculate the **average of each metric across all evaluated pairs**.

> Nothing is saved to CSV. Everything is displayed directly in Colab.



# 1. Install packages

Use a fresh Google Colab runtime with **T4 GPU or stronger**.

Do not manually install or upgrade NumPy/Pandas.


In [5]:
%pip install -q \
    --upgrade-strategy only-if-needed \
    "transformers>=4.53.0" \
    datasets accelerate bitsandbytes \
    sacrebleu bert-score \
    google-genai huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00


# 2. Imports

In [6]:
import re
import gc
import random
import getpass
import string
import unicodedata
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from tqdm.auto import tqdm
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable GPU in Colab: Runtime → Change runtime type → GPU")

GPU available: True
GPU: Tesla T4


# 3. Load the dataset

In [7]:
DATASET_NAME = "Bilash911/BanglaNewsDataset"

dataset = load_dataset(DATASET_NAME, split="train")
df = dataset.to_pandas()

df = df.dropna(subset=["Incident", "news body", "news source"]).copy()

for col in ["Incident", "news headline", "news body", "news source", "news link"]:
    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df = df[
    (df["Incident"] != "")
    & (df["news body"] != "")
    & (df["news source"] != "")
].copy()

df = df.drop_duplicates(
    subset=["Incident", "news source", "news headline", "news body"]
).reset_index(drop=True)

print("Total usable articles:", len(df))
print("Unique incidents:", df["Incident"].nunique())
print("Unique news sources:", df["news source"].nunique())

display(df[["Incident", "news source", "news headline"]].head(10))

newspaperDataset.csv:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/260 [00:00<?, ? examples/s]

Total usable articles: 230
Unique incidents: 13
Unique news sources: 26


,Incident,news source,news headline
0,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,জাগো নিউজ ২৪,জুলাই গণঅভ্যুত্থান ইতিহাসের অনন্য ঘটনা: নাহিদ
1,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,সময় সংবাদ,ঐতিহাসিক জুলাই গণ-অভ্যুত্থান দিবস আজ
2,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,বাংলানিউজ২৪,জুলাই গণঅভ্যুত্থান ২০২৪–এর শহীদদের গেজেট প্রকাশ
3,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,এনটিভি অনলাইন,জুলাই অভ্যুত্থান : যেসব কারণে ফুঁসে উঠেছিল মানুষ
4,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,সমকাল,ক্যাম্পাস থেকেই সূচনা গণঅভ্যুত্থানের
5,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,ইত্তেফাক,কেন অনিবার্য হয়ে উঠেছিল জুলাই গণঅভ্যুত্থান
6,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,বাংলা ট্রিবিউন,জুলাই গণঅভ্যুত্থানের বিজয়: এখন রাষ্ট্র সংস্কা...
7,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,কালের কণ্ঠ,জুলাই অভ্যুত্থান স্মরণে ছাত্রদলের মাসব্যাপী কর...
8,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,যুগান্তর,কোটা সংস্কার আন্দোলন থেকে গণ-অভ্যুত্থান
9,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,ঢাকা পোস্ট,"ছাত্র আন্দোলন থেকে গণবিস্ফোরণ, যে কারণে ১৬ বছর..."



# 4. Incident grouping
This shows how many articles and different news portals exist for each incident.


In [8]:
incident_summary = (
    df.groupby("Incident")
      .agg(
          Articles=("news body", "count"),
          Sources=("news source", "nunique")
      )
      .reset_index()
      .sort_values(["Sources", "Articles"], ascending=False)
      .reset_index(drop=True)
)

display(incident_summary)

,Incident,Articles,Sources
0,বৈশ্বিক জ্বালানি সংকট ও বিশ্ববিদ্যালয় বন্ধের ...,20,13
1,‘জুলাই সনদ’ ও সাংবিধানিক গণভোট (২০২৬),20,13
2,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,16,13
3,শেখ হাসিনা সরকারের পতন ও দেশত্যাগ (৫ আগস্ট ২০২৪),18,12
4,সাবেক প্রধানমন্ত্রী বেগম খালেদা জিয়ার প্রয়াণ,19,10
5,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটন...,18,10
6,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হ...,17,10
7,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ...",20,9
8,ডিজিটাল নিরাপত্তা আইন (DSA) ও সাইবার নিরাপত্তা...,20,9
9,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),19,8



# 5. Pick one article per portal for each incident
If a portal has multiple articles about the same incident, keep its **longest article**.


In [9]:
temp = df.copy()
temp["body_length"] = temp["news body"].str.len()

representatives = (
    temp.sort_values("body_length", ascending=False)
        .drop_duplicates(subset=["Incident", "news source"])
        .reset_index(drop=True)
)

display(
    representatives[
        ["Incident", "news source", "news headline", "body_length"]
    ]
)

,Incident,news source,news headline,body_length
0,শেখ হাসিনা সরকারের পতন ও দেশত্যাগ (৫ আগস্ট ২০২৪),প্রথম আলো,"ছাত্র–জনতার বিজয়, শেখ হাসিনার বিদায়",11822
1,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,আরটিভি অনলাইন,কোটা সংস্কার আন্দোলন থেকে সরকার পতন,11352
2,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হ...,জাগো নিউজ ২৪,২০২৫ সালে আন্তর্জাতিক অপরাধ ট্রাইব্যুনালে বিচা...,10438
3,দেশজুড়ে হামের প্রাদুর্ভাব (২০২৬),জাগো নিউজ ২৪,বাংলাদেশে হামের মহামারি এবং একটি জনস্বাস্থ্য চ...,10400
4,‘জুলাই সনদ’ ও সাংবিধানিক গণভোট (২০২৬),প্রথম আলো,"জুলাই সনদ ও গণভোট: ঐকমত্য, নাকি চাপিয়ে দেওয়া...",9970
...,...,...,...,...
124,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),ইত্তেফাক,ভেঙে দেওয়া হলো অন্তর্বর্তী সরকার,742
125,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হ...,বাংলানিউজ২৪,মানবতাবিরোধী অপরাধ: শেখ হাসিনার মামলার রায় ১৭...,698
126,শেখ হাসিনা সরকারের পতন ও দেশত্যাগ (৫ আগস্ট ২০২৪),যমুনা টিভি,পদত্যাগ করে দেশ ছেড়েছেন শেখ হাসিনা,670
127,সাবেক প্রধানমন্ত্রী বেগম খালেদা জিয়ার প্রয়াণ,দেশ রূপান্তর,খালেদা জিয়ার মৃত্যুতে গুলশানে শোকের ছায়া,269



# 6. Create different-newspaper pairs

Only articles from the **same incident** but **different sources** are paired.

Default:
```python
PAIRS_PER_INCIDENT = 2
```
This keeps the notebook practical and balanced.

Use:
```python
PAIRS_PER_INCIDENT = None
```
to evaluate **every possible cross-source pair**.


In [10]:
PAIRS_PER_INCIDENT = 2

all_pairs = []

for incident, group in representatives.groupby("Incident"):
    articles = group.to_dict("records")
    possible_pairs = list(combinations(articles, 2))

    possible_pairs = [
        (a, b)
        for a, b in possible_pairs
        if a["news source"] != b["news source"]
    ]

    random.Random(SEED).shuffle(possible_pairs)

    if PAIRS_PER_INCIDENT is not None:
        possible_pairs = possible_pairs[:PAIRS_PER_INCIDENT]

    for a, b in possible_pairs:
        all_pairs.append({
            "Incident": incident,
            "source_a": a["news source"],
            "headline_a": a["news headline"],
            "body_a": a["news body"],
            "source_b": b["news source"],
            "headline_b": b["news headline"],
            "body_b": b["news body"],
        })

pairs = pd.DataFrame(all_pairs).reset_index(drop=True)
pairs.insert(0, "pair_id", range(1, len(pairs) + 1))

print("Total pairs selected:", len(pairs))

display(
    pairs[
        ["pair_id", "Incident", "source_a", "headline_a", "source_b", "headline_b"]
    ]
)

Total pairs selected: 26


,pair_id,Incident,source_a,headline_a,source_b,headline_b
0,1,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),সময় সংবাদ,"কোথায় আছেন ড. মুহাম্মদ ইউনূস, করছেন কী?",আরটিভি অনলাইন,অন্তর্বর্তী সরকারের ১ বছর পূর্তি আজ
1,2,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),ঢাকা পোস্ট,"নবীন-প্রবীণে অন্তর্বর্তী সরকার, কার কী পরিচয়?",জাগো নিউজ ২৪,অন্তর্বর্তী সরকারের শপথ বৈধ: আপিল বিভাগ
2,3,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ...",যুগান্তর,আইএমএফের ঋণশর্ত ভোগান্তি বাড়াচ্ছে,বাংলা ট্রিবিউন,রিজার্ভেও আইএমএফের শর্ত পূরণ করলো বাংলাদেশ
3,4,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ...",যুগান্তর,আইএমএফের ঋণশর্ত ভোগান্তি বাড়াচ্ছে,সময় সংবাদ,আইএমএফের ঋণের তৃতীয় কিস্তির ১১৫ কোটি ডলার পেল ...
4,5,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হ...,সময় সংবাদ,মানবতাবিরোধী অপরাধ: এক নজরে শেখ হাসিনাসহ ৩ আসা...,প্রথম আলো,"শেখ হাসিনার বিরুদ্ধে মামলার রায় কবে, জানা যাব..."
5,6,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হ...,সময় সংবাদ,মানবতাবিরোধী অপরাধ: এক নজরে শেখ হাসিনাসহ ৩ আসা...,বাংলা ট্রিবিউন,আন্তর্জাতিক সংবাদমাধ্যমে শেখ হাসিনার মামলার রায়
6,7,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটন...,প্রথম আলো,মাইলস্টোনে বিমান বিধ্বস্ত: ২১ জুলাইকে শিক্ষাপ্...,ঢাকা পোস্ট,বিমান বিধ্বস্তের সময় মাইলস্টোনে চলছিল ক্লাস
7,8,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটন...,প্রথম আলো,মাইলস্টোনে বিমান বিধ্বস্ত: ২১ জুলাইকে শিক্ষাপ্...,দেশ রূপান্তর,উত্তরায় প্রশিক্ষণ বিমান বিধ্বস্ত
8,9,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,এনটিভি অনলাইন,জুলাই অভ্যুত্থান : যেসব কারণে ফুঁসে উঠেছিল মানুষ,জাগো নিউজ ২৪,জুলাই গণঅভ্যুত্থান ইতিহাসের অনন্য ঘটনা: নাহিদ
9,10,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,এনটিভি অনলাইন,জুলাই অভ্যুত্থান : যেসব কারণে ফুঁসে উঠেছিল মানুষ,কালের কণ্ঠ,জুলাই অভ্যুত্থান স্মরণে ছাত্রদলের মাসব্যাপী কর...


# 7. One common prompt for all models

In [11]:
MAX_BODY_CHARS = 4500

def clean_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def make_prompt(row):
    body_a = clean_text(row["body_a"])[:MAX_BODY_CHARS]
    body_b = clean_text(row["body_b"])[:MAX_BODY_CHARS]

    return f"""
তুমি একজন নিরপেক্ষ বাংলা সংবাদ বিশ্লেষক।

নিচে একই ঘটনার ওপর দুইটি ভিন্ন সংবাদমাধ্যমের প্রতিবেদন দেওয়া হয়েছে।
শুধু প্রদত্ত লেখার ভিত্তিতে তুলনা করবে। কোনো অপ্রমাণিত উদ্দেশ্য বা রাজনৈতিক পক্ষপাত অনুমান করবে না।

ঘটনা:
{row['Incident']}

সংবাদমাধ্যম A: {row['source_a']}
শিরোনাম A: {row['headline_a']}
প্রতিবেদন A:
{body_a}

সংবাদমাধ্যম B: {row['source_b']}
শিরোনাম B: {row['headline_b']}
প্রতিবেদন B:
{body_b}

একই ঘটনাকে দুই সংবাদমাধ্যম কীভাবে ভিন্নভাবে উপস্থাপন করেছে তা তুলনা করো।

ঠিক এই ৬টি অংশে উত্তর দাও:

A-এর উপস্থাপন:
B-এর উপস্থাপন:
সাদৃশ্য:
ভিন্ন গুরুত্ব/ফ্রেমিং:
টোন ও আলোচিত ব্যক্তি/পক্ষ:
সারসংক্ষেপ:

প্রতিটি অংশ ১-২টি সংক্ষিপ্ত বাক্যে লিখবে। শুধু বাংলায় উত্তর দেবে।
""".strip()


# 8. Gemini reference generation
Gemini is treated as the **reference** according to the faculty instruction.


In [12]:
from google import genai
from google.genai import types

GEMINI_MODEL = "gemini-3.1-flash-lite"

GEMINI_API_KEY = getpass.getpass("Gemini API key: ").strip()
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

def gemini_answer(prompt):
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(max_output_tokens=700)
    )
    return response.text.strip()

Gemini API key: ··········


In [13]:
gemini_outputs = []

for _, row in tqdm(pairs.iterrows(), total=len(pairs), desc="Gemini"):
    gemini_outputs.append(gemini_answer(make_prompt(row)))

results = pairs.copy()
results["Gemini Reference"] = gemini_outputs

print("Gemini reference generation complete.")
display(
    results[
        ["pair_id", "Incident", "source_a", "source_b", "Gemini Reference"]
    ]
)

Gemini:   0%|          | 0/26 [00:00<?, ?it/s]

Gemini reference generation complete.


,pair_id,Incident,source_a,source_b,Gemini Reference
0,1,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),সময় সংবাদ,আরটিভি অনলাইন,একজন নিরপেক্ষ সংবাদ বিশ্লেষক হিসেবে প্রদত্ত প্...
1,2,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),ঢাকা পোস্ট,জাগো নিউজ ২৪,প্রদত্ত প্রতিবেদনের ভিত্তিতে তুলনামূলক বিশ্লেষ...
2,3,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ...",যুগান্তর,বাংলা ট্রিবিউন,আপনার অনুরোধ অনুযায়ী দুটি সংবাদপত্রের প্রতিবেদ...
3,4,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ...",যুগান্তর,সময় সংবাদ,প্রদত্ত প্রতিবেদন দুটির ভিত্তিতে বিশ্লেষণ নিচে...
4,5,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হ...,সময় সংবাদ,প্রথম আলো,প্রদত্ত প্রতিবেদন দুটির ভিত্তিতে নিরপেক্ষ বিশ্...
5,6,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হ...,সময় সংবাদ,বাংলা ট্রিবিউন,নিরপেক্ষ সংবাদ বিশ্লেষক হিসেবে প্রদত্ত প্রতিবে...
6,7,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটন...,প্রথম আলো,ঢাকা পোস্ট,নিরপেক্ষ সংবাদ বিশ্লেষক হিসেবে প্রদত্ত প্রতিবে...
7,8,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটন...,প্রথম আলো,দেশ রূপান্তর,প্রদত্ত প্রতিবেদন দুটির ভিত্তিতে বিশ্লেষণ নিচে...
8,9,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,এনটিভি অনলাইন,জাগো নিউজ ২৪,প্রদত্ত প্রতিবেদন দুটির ভিত্তিতে বিশ্লেষণ নিচে...
9,10,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,এনটিভি অনলাইন,কালের কণ্ঠ,নিরপেক্ষ সংবাদ বিশ্লেষক হিসেবে প্রদত্ত প্রতিবে...


# 9. Qwen zero-shot

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

QWEN_MODEL = "Qwen/Qwen3-1.7B"

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    quantization_config=quantization,
    device_map="auto"
).eval()

def qwen_answer(prompt):
    messages = [{"role": "user", "content": prompt}]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=6000
    )

    device = next(qwen_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        output = qwen_model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [15]:
qwen_outputs = []

for _, row in tqdm(results.iterrows(), total=len(results), desc="Qwen"):
    qwen_outputs.append(qwen_answer(make_prompt(row)))

results["Qwen Zero-Shot"] = qwen_outputs

print("Qwen zero-shot complete.")
display(
    results[
        ["pair_id", "Incident", "source_a", "source_b", "Qwen Zero-Shot"]
    ]
)

Qwen:   0%|          | 0/26 [00:00<?, ?it/s]

Qwen zero-shot complete.


,pair_id,Incident,source_a,source_b,Qwen Zero-Shot
0,1,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),সময় সংবাদ,আরটিভি অনলাইন,য়ারিতে অনুষ্ঠিত হবে। এর পর সরকার নির্বাচন কমি...
1,2,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),ঢাকা পোস্ট,জাগো নিউজ ২৪,্রান্ত রায়ের বিরুদ্ধে একটি আপিল করা হয়েছে। এ...
2,3,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ...",যুগান্তর,বাংলা ট্রিবিউন,"�, বাংলাদেশ ব্যাংক এর রিজার্ভ সংরক্ষণ প্রতিদিন..."
3,4,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ...",যুগান্তর,সময় সংবাদ,ালে বাংলাদেশ ব্যাংক এর রিজার্ভ ছিল ২৫.৫ বিলিয়...
4,5,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হ...,সময় সংবাদ,প্রথম আলো,রাজসাক্ষ্য) হিসেবে সাক্ষ্য দিয়েছেন। এ মামলার ...
5,6,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হ...,সময় সংবাদ,বাংলা ট্রিবিউন,নের ঘটনায় মৃত্যুদণ্ড দেওয়া হয়েছে’ শিরোনামে ...
6,7,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটন...,প্রথম আলো,ঢাকা পোস্ট,"িষয়ে একটি স্থানীয় বাসিন্দা বলেন, এ ঘটনায় স্..."
7,8,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটন...,প্রথম আলো,দেশ রূপান্তর,�ক্ষিপ্ত করে দেখেছি। একটি বিমান একটি বাস থেকে ...
8,9,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,এনটিভি অনলাইন,জাগো নিউজ ২৪,"একটি সম্প্রচার উপদেষ্টা বলেছেন, একটি সম্প্রচার..."
9,10,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,এনটিভি অনলাইন,কালের কণ্ঠ,"্তিতে বলা হয়েছিল, ছাত্রদল একটি মাসব্যাপী কর্ম..."


In [16]:
del qwen_model
gc.collect()
torch.cuda.empty_cache()


# 10. Gemma zero-shot

Before running:
1. Open `google/gemma-3-4b-it` on Hugging Face.
2. Accept the Gemma license.
3. Paste your Hugging Face token.


In [17]:
from huggingface_hub import login
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

GEMMA_MODEL = "google/gemma-3-4b-it"

HF_TOKEN = getpass.getpass("Hugging Face token: ").strip()
login(token=HF_TOKEN)

gemma_processor = AutoProcessor.from_pretrained(
    GEMMA_MODEL,
    token=HF_TOKEN
)

gemma_model = Gemma3ForConditionalGeneration.from_pretrained(
    GEMMA_MODEL,
    token=HF_TOKEN,
    quantization_config=quantization,
    device_map="auto"
).eval()

def gemma_answer(prompt):
    messages = [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}]
        }
    ]

    inputs = gemma_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=6000
    )

    device = next(gemma_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output = gemma_model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False
        )

    new_tokens = output[0][input_length:]
    return gemma_processor.decode(new_tokens, skip_special_tokens=True).strip()

Hugging Face token: ··········


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [18]:
gemma_outputs = []

for _, row in tqdm(results.iterrows(), total=len(results), desc="Gemma"):
    gemma_outputs.append(gemma_answer(make_prompt(row)))

results["Gemma Zero-Shot"] = gemma_outputs

print("Gemma zero-shot complete.")
display(
    results[
        ["pair_id", "Incident", "source_a", "source_b", "Gemma Zero-Shot"]
    ]
)

Gemma:   0%|          | 0/26 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, n

Gemma zero-shot complete.


,pair_id,Incident,source_a,source_b,Gemma Zero-Shot
0,1,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),সময় সংবাদ,আরটিভি অনলাইন,এখানে দুটি সংবাদমাধ্যমের প্রতিবেদনগুলোর একটি ত...
1,2,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),ঢাকা পোস্ট,জাগো নিউজ ২৪,এখানে দুটি সংবাদমাধ্যমের প্রতিবেদন থেকে প্রাপ্...
2,3,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ...",যুগান্তর,বাংলা ট্রিবিউন,এখানে দুটি সংবাদমাধ্যমের প্রতিবেদন থেকে প্রাপ্...
3,4,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ...",যুগান্তর,সময় সংবাদ,এখানে যুগান্তর ও সময়সংবাদ - এই দুটি সংবাদমাধ্য...
4,5,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হ...,সময় সংবাদ,প্রথম আলো,এখানে দুটি সংবাদমাধ্যমের প্রতিবেদন থেকে প্রাপ্...
5,6,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হ...,সময় সংবাদ,বাংলা ট্রিবিউন,এখানে দুটি সংবাদমাধ্যমের প্রতিবেদন থেকে প্রাপ্...
6,7,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটন...,প্রথম আলো,ঢাকা পোস্ট,এখানে দুটি সংবাদমাধ্যমের প্রতিবেদন থেকে ঘটনার ...
7,8,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটন...,প্রথম আলো,দেশ রূপান্তর,এখানে দুটি সংবাদমাধ্যমের প্রতিবেদন থেকে ঘটনার ...
8,9,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,এনটিভি অনলাইন,জাগো নিউজ ২৪,এখানে দুটি সংবাদমাধ্যমের প্রতিবেদন থেকে প্রাপ্...
9,10,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২...,এনটিভি অনলাইন,কালের কণ্ঠ,এখানে দুটি সংবাদমাধ্যমের প্রতিবেদন থেকে ঘটনার ...


In [19]:
del gemma_model
del gemma_processor
gc.collect()
torch.cuda.empty_cache()


# 11. Show all model outputs together
This is the main qualitative comparison table.


In [20]:
pd.set_option("display.max_colwidth", 250)
pd.set_option("display.max_rows", None)

display(
    results[
        [
            "pair_id",
            "Incident",
            "source_a",
            "source_b",
            "Gemini Reference",
            "Qwen Zero-Shot",
            "Gemma Zero-Shot"
        ]
    ]
)

,pair_id,Incident,source_a,source_b,Gemini Reference,Qwen Zero-Shot,Gemma Zero-Shot
0,1,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),সময় সংবাদ,আরটিভি অনলাইন,"একজন নিরপেক্ষ সংবাদ বিশ্লেষক হিসেবে প্রদত্ত প্রতিবেদন দুটির তুলনামূলক বিশ্লেষণ নিচে দেওয়া হলো:\n\n**A-এর উপস্থাপন:**\nপ্রতিবেদনটি ড. মুহাম্মদ ইউনূসের ক্ষমতা হস্তান্তরের পরবর্তী ব্যক্তিগত জীবন, তার দৈনন্দিন রুটিন এবং ভবিষ্যৎ পরিকল্পনার ওপর আলোকপাত...",য়ারিতে অনুষ্ঠিত হবে। এর পর সরকার নির্বাচন কমিশন এবং সরকার মন্ত্রিসভা এবং সরকার স্বৈরশাসন এবং সরকার জাতীয় সংসদ নির্বাচন প্রক্রিয়ার মধ্য দিয়ে সব খাতে সম্মিলিত হবে। এর পর সরকার স্বৈরশাসন এবং সরকার জাতীয় সংসদ নির্বাচন প্রক্রিয়ার মধ্য দিয়ে সব খ...,"এখানে দুটি সংবাদমাধ্যমের প্রতিবেদনগুলোর একটি তুলনামূলক বিশ্লেষণ দেওয়া হলো:\n\nA-এর উপস্থাপন:\nড. মুহাম্মদ ইউনূস যমুনায় পরিবার ও ভবিষ্যৎ পরিকল্পনা নিয়ে সময় কাটাচ্ছেন, যা তার ব্যক্তিগত জীবন এবং সাংগঠনিক কাজকর্মের উপর বেশি জোর দিচ্ছে। তিনি পাসপোর্ট ..."
1,2,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),ঢাকা পোস্ট,জাগো নিউজ ২৪,"প্রদত্ত প্রতিবেদনের ভিত্তিতে তুলনামূলক বিশ্লেষণ নিচে দেওয়া হলো:\n\n**A-এর উপস্থাপন:**\nপ্রতিবেদনটি ৮ আগস্ট ২০২৪-এ অন্তর্বর্তী সরকার গঠনের মুহূর্ত ও নবনিযুক্ত উপদেষ্টাদের পরিচয় তুলে ধরার ওপর আলোকপাত করেছে। এতে উপদেষ্টাদের ব্যক্তিগত জীবন, কর্মজীবন ...",্রান্ত রায়ের বিরুদ্ধে একটি আপিল করা হয়েছে। এ আপিল বিভাগের আইনজীবী আইন বিভাগের অধ্যাপক ড. আসিফ নজরুল এবং আইন বিভাগের অধ্যাপক ড. আসিফ নজরুল একজন আইনজীবী এবং সাবেক অ্যাটর্নি জেনারেল হাসান আরিফ এবং আইন বিভাগের অধ্যাপক ড. আসিফ নজরুল একজন আইনজীবী এবং...,এখানে দুটি সংবাদমাধ্যমের প্রতিবেদন থেকে প্রাপ্ত তথ্যের ওপর ভিত্তি করে একটি বিশ্লেষণ দেওয়া হলো:\n\nA-এর উপস্থাপন:\nঢাকা পোস্ট প্রতিবেদনটিতে অন্তর্বর্তীকালীন সরকারের উপদেষ্টাদের পরিচয় এবং তাদের পটভূমি সম্পর্কে বিস্তারিত তথ্য দিয়েছে। এখানে ড. ইউনূসে...
2,3,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ (২০২২-২০২৪)",যুগান্তর,বাংলা ট্রিবিউন,আপনার অনুরোধ অনুযায়ী দুটি সংবাদপত্রের প্রতিবেদনের তুলনামূলক বিশ্লেষণ নিচে দেওয়া হলো:\n\n**A-এর উপস্থাপন:**\nযুগান্তর আইএমএফ-এর শর্তপূরণকে সাধারণ মানুষের জীবনযাত্রার ব্যয় বৃদ্ধি ও অর্থনৈতিক ভোগান্তির কারণ হিসেবে উপস্থাপন করেছে।\n\n**B-এর উপস্থাপন:...,"�, বাংলাদেশ ব্যাংক এর রিজার্ভ সংরক্ষণ প্রতিদিন প্রায় ২০০ কোটি ডলার হিসাবে হারিয়ে যায়। এ প্রায় ২০০ কোটি ডলার হিসাবে বাংলাদেশ ব্যাংক এর রিজার্ভ সংরক্ষণ প্রতিদিন হারিয়ে যায়। এ প্রায় ২০০ কোটি ডলার হিসাবে বাংলাদেশ ব্যাংক এর রিজার্ভ সংরক্ষণ প্রত...",এখানে দুটি সংবাদমাধ্যমের প্রতিবেদন থেকে প্রাপ্ত তথ্যের ওপর ভিত্তি করে একটি বিশ্লেষণ দেওয়া হলো:\n\nA-এর উপস্থাপন:\nযুগান্তর মূলত আইএমএফের ঋণের শর্তগুলোর নেতিবাচক প্রভাব এবং সরকারের ওপর চাপ নিয়ে একটি সমালোচনামূলক প্রতিবেদন দিয়েছে। এখানে অর্থনৈতিক স...
3,4,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ (২০২২-২০২৪)",যুগান্তর,সময় সংবাদ,প্রদত্ত প্রতিবেদন দুটির ভিত্তিতে বিশ্লেষণ নিচে দেওয়া হলো:\n\n**A-এর উপস্থাপন:**\nযুগান্তর আইএমএফের ঋণকে একটি শোষণমূলক ব্যবস্থা হিসেবে চিহ্নিত করে এর শর্তগুলোর নেতিবাচক অর্থনৈতিক প্রভাব ও সাধারণ মানুষের ভোগান্তির ওপর গুরুত্বারোপ করেছে।\n\n**B-এর উ...,"ালে বাংলাদেশ ব্যাংক এর রিজার্ভ ছিল ২৫.৫ বিলিয়ন ডলার। এ বৃহস্পতিবার মুখপাত্র বলেন, এ রিজার্ভ বাড়ানোর প্রস্তাব করা হয়েছিল বাংলাদেশ ব্যাংকের মুখপাত্র এবং বাংলাদেশ ব্যাংকের সাথে সামঞ্জন্য প্রতিষ্ঠানগুলোর মাধ্যমে এ রিজার্ভ বাড়ানোর প্রস্তাব করা হয়...","এখানে যুগান্তর ও সময়সংবাদ - এই দুটি সংবাদমাধ্যমের প্রতিবেদন থেকে প্রাপ্ত তথ্য বিশ্লেষণ করে উত্তর দেওয়া হলো:\n\nA-এর উপস্থাপন:\nযুগান্তর আইএমএফের ঋণের শর্তগুলোর নেতিবাচক প্রভাব এবং সরকারের ওপর চাপ নিয়ে বিশেষভাবে সরব। এখানে অর্থনৈতিক সংকট, মুদ্রাস্..."
4,5,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হাসিনার বিচার,সময় সংবাদ,প্রথম আলো,"প্রদত্ত প্রতিবেদন দুটির ভিত্তিতে নিরপেক্ষ বিশ্লেষণ নিচে দেওয়া হলো:\n\n**A-এর উপস্থাপন:** প্রতিবেদনটি মামলার বিস্তারিত প্রেক্ষাপট, সাক্ষীদের বিবরণ, নথিপত্রের পরিসংখ্যান এবং যুক্তিতর্ক উপস্থাপনের প্রক্রিয়ার ওপর অধিক গুরুত্ব দিয়েছে।\n\n**B-এর উপস্থ...",রাজসাক্ষ্য) হিসেবে সাক্ষ্য দিয়েছেন। এ মামলার প্রসিকিউশন পক্ষে যুক্তিতর্ক শেষ হয়েছে। এ মামলার রায় কবে দেওয়া হবে আজ বৃহস্পতিবার। এ মামলার রায় কবে দেওয়া হবে আজ বৃহস্পতিবার। এ মামলার রায় কবে দেওয়া হবে আজ বৃহস্পতিবার। এ মামলার রায় কবে দেওয়া ...,এখ


# 12. Evaluation metrics

For **every pair**:
- Qwen vs Gemini
- Gemma vs Gemini

Metrics:
- BLEU (0–100)
- BERTScore P/R/F1
- ROUGE-1/2/L F1


In [21]:
from sacrebleu import sentence_bleu
from bert_score import score as bert_score

PUNCT = string.punctuation + "।॥“”‘’…–—«»"

def tokens(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"\s+", " ", text).strip()

    output = []
    for word in text.split():
        word = word.strip(PUNCT)
        if word:
            output.append(word)
    return output

def ngrams(words, n):
    return [
        tuple(words[i:i+n])
        for i in range(len(words) - n + 1)
    ]

def overlap_f1(overlap, pred_count, ref_count):
    if overlap == 0 or pred_count == 0 or ref_count == 0:
        return 0.0

    precision = overlap / pred_count
    recall = overlap / ref_count
    return 2 * precision * recall / (precision + recall)

def rouge_n(prediction, reference, n):
    pred = Counter(ngrams(tokens(prediction), n))
    ref = Counter(ngrams(tokens(reference), n))

    overlap = sum((pred & ref).values())

    return overlap_f1(
        overlap,
        sum(pred.values()),
        sum(ref.values())
    )

def lcs_length(a, b):
    previous = [0] * (len(b) + 1)

    for x in a:
        current = [0]
        for j, y in enumerate(b, start=1):
            if x == y:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(current[-1], previous[j]))
        previous = current

    return previous[-1]

def rouge_l(prediction, reference):
    pred = tokens(prediction)
    ref = tokens(reference)

    if not pred or not ref:
        return 0.0

    lcs = lcs_length(pred, ref)
    return overlap_f1(lcs, len(pred), len(ref))

def bleu(prediction, reference):
    return sentence_bleu(
        prediction,
        [reference],
        tokenize="none",
        smooth_method="exp"
    ).score

# 13. Calculate every pair's scores

In [22]:
metric_rows = []
reference = results["Gemini Reference"].tolist()

for model_name in ["Qwen Zero-Shot", "Gemma Zero-Shot"]:

    predictions = results[model_name].tolist()

    P, R, F1 = bert_score(
        predictions,
        reference,
        model_type="bert-base-multilingual-cased",
        batch_size=8,
        verbose=False
    )

    for i, row in results.iterrows():
        pred = predictions[i]
        ref = reference[i]

        metric_rows.append({
            "pair_id": row["pair_id"],
            "Incident": row["Incident"],
            "Sources": f"{row['source_a']} vs {row['source_b']}",
            "Model": model_name,
            "BLEU": bleu(pred, ref),
            "BERTScore_P": float(P[i]),
            "BERTScore_R": float(R[i]),
            "BERTScore_F1": float(F1[i]),
            "ROUGE_1_F1": rouge_n(pred, ref, 1),
            "ROUGE_2_F1": rouge_n(pred, ref, 2),
            "ROUGE_L_F1": rouge_l(pred, ref)
        })

scores = pd.DataFrame(metric_rows)

display(scores.round(4))

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,pair_id,Incident,Sources,Model,BLEU,BERTScore_P,BERTScore_R,BERTScore_F1,ROUGE_1_F1,ROUGE_2_F1,ROUGE_L_F1
0,1,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),সময় সংবাদ vs আরটিভি অনলাইন,Qwen Zero-Shot,0.2666,0.6061,0.5423,0.5724,0.0292,0.0000,0.0292
1,2,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),ঢাকা পোস্ট vs জাগো নিউজ ২৪,Qwen Zero-Shot,0.3780,0.5667,0.5497,0.5580,0.0537,0.0068,0.0403
2,3,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ (২০২২-২০২৪)",যুগান্তর vs বাংলা ট্রিবিউন,Qwen Zero-Shot,0.3109,0.5885,0.5203,0.5523,0.0244,0.0000,0.0244
3,4,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ (২০২২-২০২৪)",যুগান্তর vs সময় সংবাদ,Qwen Zero-Shot,0.4358,0.6210,0.5607,0.5893,0.0389,0.0078,0.0389
4,5,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হাসিনার বিচার,সময় সংবাদ vs প্রথম আলো,Qwen Zero-Shot,0.3404,0.5482,0.5617,0.5548,0.0909,0.0070,0.0629
5,6,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হাসিনার বিচার,সময় সংবাদ vs বাংলা ট্রিবিউন,Qwen Zero-Shot,0.5743,0.6034,0.5911,0.5972,0.0876,0.0074,0.0657
6,7,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটনা (২০২৫),প্রথম আলো vs ঢাকা পোস্ট,Qwen Zero-Shot,0.2686,0.5718,0.5376,0.5541,0.0222,0.0000,0.0222
7,8,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটনা (২০২৫),প্রথম আলো vs দেশ রূপান্তর,Qwen Zero-Shot,0.2950,0.5774,0.5415,0.5589,0.0160,0.0000,0.0160
8,9,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২০২৪),এনটিভি অনলাইন vs জাগো নিউজ ২৪,Qwen Zero-Shot,0.5969,0.6158,0.5509,0.5815,0.1061,0.0082,0.0898
9,10,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২০২৪),এনটিভি অনলাইন vs কালের কণ্ঠ,Qwen Zero-Shot,0.2908,0.5946,0.5378,0.5648,0.0961,0.0000,0.0786



# 14. FINAL RESULT — Average of all pair-wise scores

This is the main table to show your faculty.

Each value is the **average of that metric across every evaluated pair**.


In [23]:
metrics = [
    "BLEU",
    "BERTScore_P",
    "BERTScore_R",
    "BERTScore_F1",
    "ROUGE_1_F1",
    "ROUGE_2_F1",
    "ROUGE_L_F1"
]

average_scores = (
    scores.groupby("Model")[metrics]
          .mean()
          .reset_index()
)

average_scores.insert(1, "Number_of_Pairs", len(results))

display(average_scores.round(4))

,Model,Number_of_Pairs,BLEU,BERTScore_P,BERTScore_R,BERTScore_F1,ROUGE_1_F1,ROUGE_2_F1,ROUGE_L_F1
0,Gemma Zero-Shot,26,3.7943,0.8015,0.7878,0.7946,0.3781,0.1409,0.2395
1,Qwen Zero-Shot,26,0.6977,0.6263,0.5781,0.6010,0.1076,0.0261,0.0787



# 15. Average score by incident
Optional, but useful to see which incidents were easier or harder.


In [24]:
incident_average = (
    scores.groupby(["Incident", "Model"])[metrics]
          .mean()
          .reset_index()
)

display(incident_average.round(4))

,Incident,Model,BLEU,BERTScore_P,BERTScore_R,BERTScore_F1,ROUGE_1_F1,ROUGE_2_F1,ROUGE_L_F1
0,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),Gemma Zero-Shot,3.7084,0.8095,0.7940,0.8016,0.4037,0.1257,0.2385
1,অন্তর্বর্তী সরকার প্রতিষ্ঠা (৮ আগস্ট ২০২৪),Qwen Zero-Shot,0.3223,0.5864,0.5460,0.5652,0.0414,0.0034,0.0347
2,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ (২০২২-২০২৪)",Gemma Zero-Shot,2.9802,0.8074,0.7898,0.7985,0.3651,0.1457,0.2539
3,"অর্থনৈতিক সংকট, ডলারের ঘাটতি ও আইএমএফ (IMF) ঋণ (২০২২-২০২৪)",Qwen Zero-Shot,0.3733,0.6047,0.5405,0.5708,0.0317,0.0039,0.0317
4,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হাসিনার বিচার,Gemma Zero-Shot,2.7914,0.7916,0.7809,0.7862,0.3669,0.1331,0.2272
5,আন্তর্জাতিক অপরাধ ট্রাইব্যুনাল (ICT) এবং শেখ হাসিনার বিচার,Qwen Zero-Shot,0.4573,0.5758,0.5764,0.5760,0.0893,0.0072,0.0643
6,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটনা (২০২৫),Gemma Zero-Shot,1.4612,0.7777,0.7653,0.7714,0.3001,0.0842,0.1963
7,উত্তরায় বিমান বাহিনীর প্রশিক্ষণ বিমান দুর্ঘটনা (২০২৫),Qwen Zero-Shot,0.2818,0.5746,0.5395,0.5565,0.0191,0.0000,0.0191
8,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২০২৪),Gemma Zero-Shot,4.0149,0.7835,0.7827,0.7831,0.3586,0.1237,0.2177
9,জুলাই হত্যাকাণ্ড ও ছাত্র-জনতার গণঅভ্যুত্থান (২০২৪),Qwen Zero-Shot,0.4439,0.6052,0.5444,0.5732,0.1011,0.0041,0.0842


# 16. Which model scored higher?

In [25]:
comparison = []

for metric in metrics:
    best_index = average_scores[metric].idxmax()

    comparison.append({
        "Metric": metric,
        "Higher Average Model": average_scores.loc[best_index, "Model"],
        "Higher Average Score": average_scores.loc[best_index, metric]
    })

display(pd.DataFrame(comparison).round(4))

,Metric,Higher Average Model,Higher Average Score
0,BLEU,Gemma Zero-Shot,3.7943
1,BERTScore_P,Gemma Zero-Shot,0.8015
2,BERTScore_R,Gemma Zero-Shot,0.7878
3,BERTScore_F1,Gemma Zero-Shot,0.7946
4,ROUGE_1_F1,Gemma Zero-Shot,0.3781
5,ROUGE_2_F1,Gemma Zero-Shot,0.1409
6,ROUGE_L_F1,Gemma Zero-Shot,0.2395



# 17. Faculty explanation

### Zero-shot
Qwen and Gemma are used **without any training or fine-tuning** on this project dataset.

### Gemini reference
Gemini's comparison is treated as the reference/ground truth according to the faculty instruction.

### Pair-wise scoring
Each Qwen and Gemma answer is compared with Gemini for the **same newspaper pair**.

### Average of all scores
If there are `N` pairs and a metric produces:

`score1, score2, ..., scoreN`

then:

**Average = (score1 + score2 + ... + scoreN) / N**

The averaging is done separately for BLEU, BERTScore and each ROUGE metric.

### Most important output
**Section 14 — FINAL RESULT — Average of all pair-wise scores**
